In [ ]:
import cv2
import mediapipe as mp
import numpy as np
print('Ready')

Ready


In [ ]:
# import mediapipe as mp
# print(mp.__version__)

In [ ]:
import time
import cv2
import mediapipe as mp
import numpy as np

# --- 1. BIOMETRIC ANGLE CALCULATOR ---
def calculate_angle(a, b, c):
    """Calculates the exact geometric interior angle at joint 'b' given three points a, b, and c."""
    a = np.array(a)  
    b = np.array(b)  
    c = np.array(c)  
    
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    
    if angle > 180.0:
        angle = 360 - angle
    return int(angle)

# --- HELPER FUNCTION: SHUTTERSTOCK-STYLE 3D TEXT EFFECT ---
def draw_3d_text(img, text, position, font, scale):
    """Draws white text with a clean, thick black drop shadow layer behind it for a 3D look."""
    x, y = position
    cv2.putText(img, text, (x + 2, y + 2), font, scale, (0, 0, 0), 6, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), font, scale, (255, 255, 255), 2, cv2.LINE_AA)

# --- HELPER FUNCTION: WRAP MULTI-LINE 3D TEXT ---
def draw_wrapped_3d_text(img, text, position, font, scale, line_spacing=40):
    """Splits long coaching sentences and applies the 3D shadow layer effect to each line."""
    x, y = position
    words = text.split(' ')
    lines = []
    current_line = []
    
    for word in words:
        current_line.append(word)
        if len(' '.join(current_line)) > 35:
            current_line.pop()
            lines.append(' '.join(current_line))
            current_line = [word]
    lines.append(' '.join(current_line))
    
    for i, line in enumerate(lines):
        if line.strip():
            draw_3d_text(img, line, (x, y + (i * line_spacing)), font, scale)

# --- Mediapipe Framework Setup ---
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7, min_tracking_confidence=0.7)

cap = cv2.VideoCapture(0)

# --- Flow Control States ---
sport_selected = False
sport = ""
next_screen = False

# --- Coaching Loop States ---
coaching_state = "MOVE_BACK"  
attempts = 0
screen_lock_time = None  

# --- Performance Analytics History Ledger ---
session_history = []  
review_comment = ""

# Live Real-Time Variable Buffers
detected_action = "Unknown"
live_form_score = 0
live_balance_score = 0

# --- Navigation Timers ---
selection_start_time = None
reset_start_time = None
coaching_timer = None

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape  

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pose_results = pose.process(rgb)
    hand_results = hands.process(rgb)

    if pose_results.pose_landmarks:
        mp_drawing.draw_landmarks(frame, pose_results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

    if hand_results.multi_hand_landmarks:
        for hand_landmarks in hand_results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            landmarks = hand_landmarks.landmark

            fingers = 0
            if landmarks[8].y < landmarks[6].y: fingers += 1   
            if landmarks[12].y < landmarks[10].y: fingers += 1 
            if landmarks[16].y < landmarks[14].y: fingers += 1 
            if landmarks[20].y < landmarks[18].y: fingers += 1 

            if next_screen and (coaching_state == "MOVE_BACK" or coaching_state == "CAPTURING"):
                continue

            if next_screen and (coaching_state == "SHOW_RESULT" or coaching_state == "SUMMARY"):
                if screen_lock_time is not None and (time.time() - screen_lock_time < 3):
                    continue  

            if next_screen and coaching_state == "SHOW_RESULT":
                if fingers == 1:  
                    attempts += 1  
                    coaching_state = "MOVE_BACK"
                    coaching_timer = time.time()
                    continue
                elif fingers == 0:  
                    coaching_state = "SUMMARY"
                    screen_lock_time = time.time()  
                    coaching_timer = time.time()
                    continue

            elif next_screen and coaching_state == "SUMMARY":
                if fingers == 1:  
                    sport = ""
                    sport_selected = False
                    next_screen = False
                    coaching_state = "MOVE_BACK"
                    attempts = 0
                    session_history = []  
                    selection_start_time = None
                    screen_lock_time = None
                    continue

            if not next_screen:
                if fingers == 1: current_choice = "Cricket"
                elif fingers == 2: current_choice = "Football"
                elif fingers == 3: current_choice = "Yoga"
                else: current_choice = ""

                if current_choice != "":
                    if sport != current_choice:
                        sport = current_choice
                        sport_selected = True
                        selection_start_time = time.time()
                    elif time.time() - selection_start_time > 5:
                        next_screen = True
                        coaching_state = "MOVE_BACK"
                        coaching_timer = time.time()
                        attempts = 1
                else:
                    sport = ""
                    sport_selected = False
                    selection_start_time = None

    if not hand_results.multi_hand_landmarks and not next_screen:
        if reset_start_time is None:
            reset_start_time = time.time()
        elif time.time() - reset_start_time > 2:
            sport = ""
            sport_selected = False
            selection_start_time = None
    else:
        reset_start_time = None

    # =============================================================
    # --- ACTIVE RUNTIME MACHINE ---
    # =============================================================
    if next_screen:
        
        if coaching_state == "MOVE_BACK":
            elapsed = time.time() - coaching_timer
            countdown = max(0, 5 - int(elapsed))
            
            draw_3d_text(frame, f"{sport} Mode Active", (50, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.0)
            draw_3d_text(frame, "Please move back into full view", (50, 115), cv2.FONT_HERSHEY_SIMPLEX, 0.7)
            draw_3d_text(frame, f"Starting in... {countdown}s", (50, 185), cv2.FONT_HERSHEY_SIMPLEX, 1.1)
            
            if elapsed > 5:
                coaching_state = "CAPTURING"
                coaching_timer = time.time()
                detected_action = "No Action Detected"
                live_form_score = 0
                live_balance_score = 0

        elif coaching_state == "CAPTURING":
            elapsed = time.time() - coaching_timer
            countdown = max(0, 5 - int(elapsed))
            
            draw_3d_text(frame, f"RECORDING ATTEMPT #{attempts}", (50, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.0)
            draw_3d_text(frame, f"Perform Action Now: {countdown}s", (50, 115), cv2.FONT_HERSHEY_SIMPLEX, 0.8)

            if pose_results.pose_landmarks:
                pts = pose_results.pose_landmarks.landmark
                
                nose = [pts[0].x * w, pts[0].y * h]
                l_shoulder = [pts[11].x * w, pts[11].y * h]
                r_shoulder = [pts[12].x * w, pts[12].y * h]
                l_elbow = [pts[13].x * w, pts[13].y * h]
                r_elbow = [pts[14].x * w, pts[14].y * h]
                l_wrist = [pts[15].x * w, pts[15].y * h]
                r_wrist = [pts[16].x * w, pts[16].y * h]
                l_hip = [pts[23].x * w, pts[23].y * h]
                r_hip = [pts[24].x * w, pts[24].y * h]
                l_knee = [pts[25].x * w, pts[25].y * h]
                r_knee = [pts[26].x * w, pts[26].y * h]
                l_ankle = [pts[27].x * w, pts[27].y * h]
                r_ankle = [pts[28].x * w, pts[28].y * h]

                left_elbow_angle = calculate_angle(l_shoulder, l_elbow, l_wrist)
                right_elbow_angle = calculate_angle(r_shoulder, r_elbow, r_wrist)
                left_knee_angle = calculate_angle(l_hip, l_knee, l_ankle)
                right_knee_angle = calculate_angle(r_hip, r_knee, r_ankle)

                # =============================================================
                # --- NEW YOGA MACHINE: POINTS-BASED PRIORITY SYSTEM ---
                # =============================================================
                if sport == "Yoga":
                    max_knee = max(left_knee_angle, right_knee_angle)
                    min_knee = min(left_knee_angle, right_knee_angle)
                    
                    tree_probability = 0
                    warrior_probability = 0
                    
                    # Geometry Check 1: Upper-Body Arm Plane Profile
                    are_hands_above_head = pts[15].y < pts[0].y and pts[16].y < pts[0].y
                    are_arms_spread_wide = abs(pts[15].x - pts[16].x) > (abs(pts[11].x - pts[12].x) * 2.5)
                    
                    if are_hands_above_head:
                        tree_probability += 50
                    if are_arms_spread_wide:
                        warrior_probability += 50
                        
                    # Geometry Check 2: Stance Expansion Matrix
                    foot_stance_width = abs(pts[27].x - pts[28].x)
                    shoulder_width = abs(pts[11].x - pts[12].x)
                    
                    if foot_stance_width > (shoulder_width * 2.0):
                        warrior_probability += 40
                    else:
                        tree_probability += 40

                    # Dynamic Routing Evaluation
                    if tree_probability >= warrior_probability:
                        detected_action = "Tree Pose"
                        # Professional Tolerance Curve: 5-degree cushion for clean scores
                        knee_error = abs(min_knee - 45)
                        live_form_score = int(100 - (knee_error * 0.8)) if knee_error > 5 else 98
                        
                        center_error = abs(pts[0].x - ((pts[23].x + pts[24].x) / 2)) * w
                        live_balance_score = int(100 - (center_error * 0.4))
                    else:
                        detected_action = "Warrior Pose"
                        # Professional Tolerance Curve for deep lunge
                        lunge_error = abs(min_knee - 90)
                        live_form_score = int(100 - (lunge_error * 0.7)) if lunge_error > 5 else 98
                        
                        rear_leg_error = abs(max_knee - 175)
                        live_balance_score = int(100 - (rear_leg_error * 0.6))

                # =============================================================
                # --- CRICKET MODULE: LOCKED DOWN ---
                # =============================================================
                elif sport == "Cricket":
                    if l_wrist[1] < l_shoulder[1] + 20:
                        detected_action = "Pull Shot"
                        live_form_score = int(max(40, 100 - abs(left_elbow_angle - 165)))
                        live_balance_score = int(max(40, 100 - abs(right_knee_angle - 135)))
                    else:
                        detected_action = "Forward Defense"
                        live_form_score = int(max(40, 100 - abs(left_elbow_angle - 95)))
                        live_balance_score = int(max(40, 100 - abs(left_knee_angle - 95)))

                # =============================================================
                # --- FOOTBALL MODULE: VERIFIED ANKLE SCALING ENGINE ---
                # =============================================================
                elif sport == "Football":
                    # Tracks screen-normalized height delta (.y inputs scale from 0.0 to 1.0)
                    ankle_height_gap = abs(pts[27].y - pts[28].y)

                    # PENALTY KICK: One ankle is lifted clearly off the ground plane
                    if ankle_height_gap > 0.03:
                        detected_action = "Penalty Kick"
                        kicking_knee = min(left_knee_angle, right_knee_angle)
                        # Professional Tolerance Curve: 6-degree cushion for elite score range
                        knee_error = abs(kicking_knee - 90)
                        live_form_score = int(100 - (knee_error * 0.8)) if knee_error > 6 else 98
                        
                        plant_error = abs(max(left_knee_angle, right_knee_angle) - 140)
                        live_balance_score = int(100 - (plant_error * 0.6))
                    
                    # GOALKEEPER SAVE: Both feet stay grounded side-by-side
                    else:
                        detected_action = "Goalkeeper Save"
                        # Professional Tolerance Curve for extended wingspan
                        left_arm_err = abs(left_elbow_angle - 175)
                        right_arm_err = abs(right_elbow_angle - 175)
                        live_form_score = int(100 - ((left_arm_err + right_arm_err) * 0.4))
                        
                        symmetry_delta = abs(left_knee_angle - right_knee_angle)
                        live_balance_score = int(100 - (symmetry_delta * 0.8))

                # Safe global bounds clamp protection layers
                live_form_score = max(0, min(100, live_form_score))
                live_balance_score = max(0, min(100, live_balance_score))

                draw_3d_text(frame, f"Tracking: {detected_action}", (50, 185), cv2.FONT_HERSHEY_SIMPLEX, 0.7)

            if elapsed > 5:
                previous_run = None
                for past in reversed(session_history):
                    if past["action"] == detected_action:
                        previous_run = past
                        break

                if previous_run is not None:
                    score_diff = live_form_score - previous_run["form"]
                    if score_diff > 0:
                        review_comment = f"Review: Great job! Your {detected_action} accuracy increased by {score_diff}%."
                    elif score_diff < 0:
                        review_comment = f"Review: Your {detected_action} form slipped by {abs(score_diff)}%. Keep posture locked."
                    else:
                        review_comment = f"Review: Consistent! You matched your previous {detected_action} score."
                else:
                    if detected_action == "Tree Pose":
                        review_comment = "Tip: Press your knee farther back out to the side to lock core stability."
                    elif detected_action == "Warrior Pose":
                        review_comment = "Tip: Drive your front hip down more to bring that leading knee closer to 90 degrees."
                    elif detected_action == "Forward Defense":
                        review_comment = "Tip: Drop your wrists low and ensure your head remains directly over the ball path line."
                    elif detected_action == "Pull Shot":
                        review_comment = "Tip: Extend your arms horizontally across your chest plane to generate power safely."
                    
                    # OPTION 2 SHORT PHRASE CONFIRMED FOR DISPLAY
                    elif detected_action == "Penalty Kick":
                        review_comment = "To get the best result: plant foot stable, chest leaned forward, kicking leg at 90°."
                        
                    elif detected_action == "Goalkeeper Save":
                        review_comment = "Tip: Explode your arms wide and keep your base balanced to block maximum net space."
                    else:
                        review_comment = "Performance base established. Balance and adjust your joints for attempt 2!"

                session_history.append({
                    "action": detected_action,
                    "form": live_form_score,
                    "balance": live_balance_score
                })
                
                coaching_state = "SHOW_RESULT"
                screen_lock_time = time.time()  

        elif coaching_state == "SHOW_RESULT":
            draw_3d_text(frame, f"--- {sport} Performance Card ---", (50, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.85)
            draw_3d_text(frame, f"Identified Action: {detected_action}", (50, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.7)
            draw_3d_text(frame, f"Calculated Form Score: {live_form_score}/100", (50, 145), cv2.FONT_HERSHEY_SIMPLEX, 0.7)
            draw_3d_text(frame, f"Calculated Balance Rating: {live_balance_score}/100", (50, 195), cv2.FONT_HERSHEY_SIMPLEX, 0.7)
            draw_3d_text(frame, f"Total Drill Sets Run: {attempts}", (50, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.7)
            
            draw_wrapped_3d_text(frame, review_comment, (50, 295), cv2.FONT_HERSHEY_SIMPLEX, 0.58)
            
            if time.time() - screen_lock_time > 3:
                draw_3d_text(frame, "Show 1 Finger to try again", (50, 395), cv2.FONT_HERSHEY_SIMPLEX, 0.6)
                draw_3d_text(frame, "Close Fist for Session Summary", (50, 435), cv2.FONT_HERSHEY_SIMPLEX, 0.6)
            else:
                draw_3d_text(frame, "Locking screen for review...", (50, 410), cv2.FONT_HERSHEY_SIMPLEX, 0.6)

        elif coaching_state == "SUMMARY":
            best_set = max(session_history, key=lambda x: x["form"]) if session_history else {"action": "None", "form": 0}
            
            draw_3d_text(frame, "--- Session Summary ---", (50, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.0)
            draw_3d_text(frame, f"Sport Category: {sport}", (50, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.8)
            draw_3d_text(frame, f"Total Attempt Cycles: {attempts}", (50, 190), cv2.FONT_HERSHEY_SIMPLEX, 0.8)
            
            draw_3d_text(frame, f"Peak Performance Action: {best_set['action']}", (50, 245), cv2.FONT_HERSHEY_SIMPLEX, 0.7)
            draw_3d_text(frame, f"Max Form Score Reached: {best_set['form']}/100", (50, 290), cv2.FONT_HERSHEY_SIMPLEX, 0.7)
            
            if time.time() - screen_lock_time > 3:
                draw_3d_text(frame, "Show 1 Finger to exit to Main Menu", (50, 365), cv2.FONT_HERSHEY_SIMPLEX, 0.7)
            else:
                draw_3d_text(frame, "Locking screen for review...", (50, 365), cv2.FONT_HERSHEY_SIMPLEX, 0.6)

    elif not sport_selected:
        draw_3d_text(frame, "Show your fingers to select a game", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7)
        draw_3d_text(frame, "1 Finger - Cricket", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.6)
        draw_3d_text(frame, "2 Fingers - Football", (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.6)
        draw_3d_text(frame, "3 Fingers - Yoga", (20, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.6)

    else:
        draw_3d_text(frame, f"Selected Sport: {sport}", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8)
        if selection_start_time is not None:
            countdown = max(0, 5 - int(time.time() - selection_start_time))
            draw_3d_text(frame, f"Holding... {countdown}s", (20, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.7)

    cv2.imshow("AI Sports Coach", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()